In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

from catboost import CatBoostClassifier

In [3]:
PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

TARGET = "임신 성공 여부"

BASELINE_VALID_AUC = 0.737416

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True


In [4]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int)

X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

X_raw = X_raw.drop(columns=id_cols, errors="ignore")
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore")

print("X_raw:", X_raw.shape)
print("y:", y.shape)
print("X_test_raw:", X_test_raw.shape)

X_raw: (256351, 67)
y: (256351,)
X_test_raw: (90067, 67)


In [5]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [6]:
def safe_ratio_minus1(df, numerator_col, denominator_col):
    """
    분모가 0이면 -1,
    아니면 numerator / denominator.
    """
    numerator = pd.to_numeric(df[numerator_col], errors="coerce").fillna(0)
    denominator = pd.to_numeric(df[denominator_col], errors="coerce").fillna(0)

    return np.where(
        denominator == 0,
        -1,
        numerator / denominator
    )


def data_preprocessing_ratio_minus1(df):
    """
    기존 champion preprocessing을 먼저 적용한 뒤,
    일부 ratio feature만 0 sentinel -> -1 sentinel로 덮어쓰기.

    배아_이식_집중도는 건드리지 않음.
    """
    df = data_preprocessing(df)

    # 1. 배아 생성률
    if {"총 생성 배아 수", "혼합된 난자 수"}.issubset(df.columns):
        df["배아_생성률"] = safe_ratio_minus1(
            df,
            numerator_col="총 생성 배아 수",
            denominator_col="혼합된 난자 수"
        )

    # 2. 배아 이식률
    if {"이식된 배아 수", "총 생성 배아 수"}.issubset(df.columns):
        df["배아_이식률"] = safe_ratio_minus1(
            df,
            numerator_col="이식된 배아 수",
            denominator_col="총 생성 배아 수"
        )

    # 3. 배아 냉동률
    if {"저장된 배아 수", "총 생성 배아 수"}.issubset(df.columns):
        df["배아_냉동률"] = safe_ratio_minus1(
            df,
            numerator_col="저장된 배아 수",
            denominator_col="총 생성 배아 수"
        )

    # 4. IVF 임신성공률
    if {"IVF 임신 횟수", "IVF 시술 횟수"}.issubset(df.columns):
        df["IVF_임신성공률"] = safe_ratio_minus1(
            df,
            numerator_col="IVF 임신 횟수",
            denominator_col="IVF 시술 횟수"
        )

    # 5. DI 임신성공률
    if {"DI 임신 횟수", "DI 시술 횟수"}.issubset(df.columns):
        df["DI_임신성공률"] = safe_ratio_minus1(
            df,
            numerator_col="DI 임신 횟수",
            denominator_col="DI 시술 횟수"
        )

    # 6. 출산 / 임신 전환율
    if {"총 출산 횟수", "총 임신 횟수"}.issubset(df.columns):
        df["출산_임신_전환율"] = safe_ratio_minus1(
            df,
            numerator_col="총 출산 횟수",
            denominator_col="총 임신 횟수"
        )

    # 7. 클리닉 집중도
    if {"클리닉 내 총 시술 횟수", "총 시술 횟수"}.issubset(df.columns):
        df["클리닉_집중도"] = safe_ratio_minus1(
            df,
            numerator_col="클리닉 내 총 시술 횟수",
            denominator_col="총 시술 횟수"
        )

    # 핵심 feature라 유지
    # df["배아_이식_집중도"]는 기존 그대로 둠

    return df

In [7]:
X_exp = data_preprocessing_ratio_minus1(X_raw)

ratio_cols = [
    "배아_생성률",
    "배아_이식률",
    "배아_냉동률",
    "IVF_임신성공률",
    "DI_임신성공률",
    "출산_임신_전환율",
    "클리닉_집중도",
    "배아_이식_집중도",  # 확인용. 얘는 기존 유지.
]

print("X_exp shape:", X_exp.shape)

display(X_exp[ratio_cols].describe())

for col in ratio_cols:
    print("\n", col)
    print("min:", X_exp[col].min())
    print("max:", X_exp[col].max())
    print("-1 count:", (X_exp[col] == -1).sum())

X_exp shape: (256351, 92)


,배아_생성률,배아_이식률,배아_냉동률,IVF_임신성공률,DI_임신성공률,출산_임신_전환율,클리닉_집중도,배아_이식_집중도
count,256351.000000,256351.000000,256351.000000,256351.000000,256351.000000,256351.000000,256351.000000,256351.000000
mean,0.318852,0.049279,-0.094442,-0.307771,-0.940892,-0.661909,0.095906,0.671339
std,0.708762,0.641817,0.552354,0.615144,0.251903,0.706228,0.910077,0.409096
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000
25%,0.200000,0.000000,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.250000
50%,0.600000,0.176471,0.000000,0.000000,-1.000000,-1.000000,0.333333,1.000000
75%,0.800000,0.400000,0.222222,0.000000,-1.000000,-1.000000,1.000000,1.000000
max,1.250000,3.000000,7.000000,1.000000,1.000000,1.000000,1.000000,1.000000



 배아_생성률
min: -1.0
max: 1.25
-1 count: 52941

 배아_이식률
min: -1.0
max: 3.0
-1 count: 59640

 배아_냉동률
min: -1.0
max: 7.0
-1 count: 59640

 IVF_임신성공률
min: -1.0
max: 1.0
-1 count: 103934

 DI_임신성공률
min: -1.0
max: 1.0
-1 count: 242464

 출산_임신_전환율
min: -1.0
max: 1.0
-1 count: 205426

 클리닉_집중도
min: -1.0
max: 1.0
-1 count: 97599

 배아_이식_집중도
min: 0.0
max: 1.0
-1 count: 0


In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_exp,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train mean:", y_train.mean())
print("y_val mean:", y_val.mean())

X_train: (205080, 92)
X_val: (51271, 92)
y_train mean: 0.2583479617710162
y_val mean: 0.2583526750014628


In [9]:
cat_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)

cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.02498214961001344,
    depth=8,
    l2_leaf_reg=18.591182129683194,
    random_strength=0.32969640414889206,
    bagging_temperature=4.535604806522509,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    class_weights=[1, 190123 / 66228],
    allow_writing_files=False
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    verbose=100
)

y_val_pred = cat_model.predict(X_val)
y_val_proba = cat_model.predict_proba(X_val)[:, 1]

valid_auc = roc_auc_score(y_val, y_val_proba)

print("f1:", f1_score(y_val, y_val_pred))
print("precision:", precision_score(y_val, y_val_pred))
print("recall:", recall_score(y_val, y_val_pred))
print("roc_auc:", valid_auc)

print("baseline:", BASELINE_VALID_AUC)
print("diff:", valid_auc - BASELINE_VALID_AUC)

0:	test: 0.7231333	best: 0.7231333 (0)	total: 207ms	remaining: 6m 54s
100:	test: 0.7339113	best: 0.7339113 (100)	total: 13.8s	remaining: 4m 19s
200:	test: 0.7360882	best: 0.7360882 (200)	total: 26.6s	remaining: 3m 57s
300:	test: 0.7366510	best: 0.7366548 (297)	total: 39.6s	remaining: 3m 43s
400:	test: 0.7368858	best: 0.7368872 (394)	total: 51.5s	remaining: 3m 25s
500:	test: 0.7370569	best: 0.7370569 (499)	total: 1m 2s	remaining: 3m 8s
600:	test: 0.7371547	best: 0.7371686 (596)	total: 1m 14s	remaining: 2m 53s
700:	test: 0.7372326	best: 0.7372364 (687)	total: 1m 26s	remaining: 2m 40s
800:	test: 0.7372760	best: 0.7372787 (796)	total: 1m 38s	remaining: 2m 27s
900:	test: 0.7372795	best: 0.7372843 (836)	total: 1m 50s	remaining: 2m 14s
1000:	test: 0.7373023	best: 0.7373249 (949)	total: 2m 3s	remaining: 2m 2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.737324925
bestIteration = 949

Shrink model to first 950 iterations.
f1: 0.5152875667812085
precision: 0.3858317995717

In [10]:
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

display(fi.head(50))

display(
    fi[fi["feature"].isin(ratio_cols)]
)

,feature,importance
41,이식된 배아 수,27.526693
84,배아_이식_집중도,23.030163
52,난자 출처,5.668146
1,시술 당시 나이,5.147303
83,고령_난자수_interaction,4.498650
65,배아 이식 경과일,3.929699
47,수집된 신선 난자 수,2.842015
80,배아_냉동률,2.785429
38,총 생성 배아 수,2.656572
43,저장된 배아 수,2.608209


,feature,importance
84,배아_이식_집중도,23.030163
80,배아_냉동률,2.785429
79,배아_이식률,1.021204
81,IVF_임신성공률,0.778115
78,배아_생성률,0.638106
89,출산_임신_전환율,0.634513
90,클리닉_집중도,0.335462
82,DI_임신성공률,0.200534
